-The data set I am going to use is from github user VorBoto <https://github.com/VorBoto/DnD_Spells> 

## The Why
-I have been playing Dungeons and Dragons for a few years. I mostly play for the goofs and just hanging out with friends, but I also love the absolutely absurd things you can do with spells. My current adventuring party plays pretty loose with the rules and we have a tendency to lean in favor of "yes and-ing" rather than, you know, logic and physics. 

-When looking for a data set for my Google Data Analytics capstone project I knew I wanted to do something fun while still creating an analysis that makes sense in the context of a business objective.


#### The Question
-After thinking about how I wanted to approach my capstone project, I had to figure out how to frame the fantasy data set I found into a business problem. While there is a lot of great data provided by the data set, there was not much that I could think of that a business would be able to use if this were a real business scenario. Eventually after thinking about it for a while, I decided to make some assumptions about the data in order to be able to ask the question, "What is the most cost efficient spell?"

In [ ]:
library(tidyverse)                                  # Set up libraries
library(readr)
library(skimr)
library(lubridate)
library(knitr)
library(janitor)
library(reshape2)
dnd_spells<-read_csv("../input/data-set-from-vorboto-on-github/DnD_spells.csv") # Name my data 
head(dnd_spells)    # To see an example of the the data frame
colnames(dnd_spells)

## The Assumptions 
-After gathering my data and sorting through it I had to answer some questions and make some assumptions:

-First, I had to decide how I would consider the cost of a spell as well as the desired outcome.

-In order to determine the desired outcome I decided damage output would need to be the end goal.

-Since Dungeons and Dragons is a dice based tabletop game there is no set number for the damage output of each spell.

-When you cast a spell you determine what type of dice and how many you roll in a success.

-So in order to come up with a quantifiable outcome for each type of spell, I could only include spells that cause damage and I would have to figure out a way to simplify the chance element for a spell being cast.


In [ ]:
dnd_spells_damage<-dnd_spells        # Removing spells that do not cause damage 
dnd_spells_damage$`Damage type`[dnd_spells_damage$`Damage type`=="NONE"]<-NA
dnd_spells_damage <- dnd_spells_damage %>% 
  drop_na() %>%
  subset(select= -c(Description))
unique_damage<-unique(dnd_spells_damage$Damage) # Damage needs to be converted
dnd_spells_damage<-dnd_spells_damage %>%  # finding unique damage values 
  rename(character_class = Class) # renaming to avoid syntax issues 
head(dnd_spells_damage)


-My eventual conclusion was to use <https://anydice.com/> in order to find the average outcome of each dice roll and use that as the base for our damage analysis


In [ ]:
dnd_spells_damage_average<- dnd_spells_damage %>%   # convert damage to average
mutate(average_damage = case_when(Damage == "1d4" ~ 2.5,
                                  Damage == "1d6" ~ 3.5,
                                  Damage == "1d8"~ 4.5,
                                  Damage == "1d10"~ 5.5,
                                  Damage == "1d12"~ 6.5,
                                  Damage == "3d6" ~10.5,
                                  Damage == "6d10" ~ 33,
                                  Damage == "4d6" ~ 14,
                                  Damage == "2d10" ~ 11,
                                  Damage == "3d10" ~ 16.5,
                                  Damage == "3d4 + 3" ~ 10.5,
                                  Damage == "5d8" ~ 22.5,
                                  Damage == "2d8" ~ 9,
                                  Damage == "4d4" ~ 10,
                                  Damage == "2d6" ~ 7,
                                  Damage == "3d8" ~ 13.5,
                                  Damage == "8d6" ~ 28,
                                  Damage == "8d8" ~ 36,
                                  Damage == "4d8" ~ 18,
                                  Damage == "4d10" ~ 22,
                                  Damage == "10d8" ~ 45,
                                  Damage == "10d6" ~ 35,
                                  Damage == "14d6" ~ 49,
                                  Damage == "6d8" ~ 27,
                                  Damage == "7d8" ~ 31.5,
                                  Damage == "12d6" ~ 42,
                                  Damage == "7d8 + 30" ~ 61.5,
                                  Damage == "7d10" ~ 38.5,
                                  Damage == "20d6" ~ 70
                                  ))
head(dnd_spells_damage_average)


#### A Quick Diversion
-I took a brief detour in order to begin visualizing some of the data.

-This first visual depicts damage output vs. the spell level.

-The visual shows a clear correlation between damage output and level.

In [ ]:
ggplot(dnd_spells_damage_average,aes(x=Level,y=average_damage,color=character_class))+
  geom_jitter(aes(color=character_class))+facet_wrap(~character_class)

-The next set of visuals displays the different schools of magic.

-I did find it interesting that the highest overall damage for nearly every class was either Evocation or Necromancy.


In [ ]:
ggplot(dnd_spells_damage_average,aes(x=Level,y=average_damage,color=School))+
  geom_jitter(aes(color=School))+facet_wrap(~character_class)

#### More Cleaning

-Next, I had to find a way to quantify the cost of an individual spell.

-When it comes to spellcasting in D&D there are several factors to the equation. Some factors I included and some I discarded.

-Materials, Level, and Damage were all factors that I believed could be assumed to have a cost.

-Range, Components, Duration, Casting Time, and Damage Type were all things that, while important, I decided would not be part of my analysis.

-Since I have already decided how to deal with simplifying Damage, the only factors I needed to consider now were how to find a consistent price for Materials and Spell Level.

-The materials were fairly simple. D&D has an extensive online community as well as plenty of number crunchers. There are many sources out there for pricing market goods and I ended up using this <https://docs.google.com/document/d/1x8FASGSluGqceaovEbrc9C4vmWUhlDPI/edit> as a basis for my material goods costs.

In [ ]:
unique_materials<-unique(dnd_spells_damage$Material) # Finding unique components

In [ ]:
dnd_spell_value_v1<- dnd_spells_damage_average %>% # putting value to items
mutate(material_cost = case_when(Material == "A bit of fur and a rod of amber, Crystal, or glass" ~ 60,
                                 Material == "A bit of fur; a piece of amber, glass, or a Crystal rod; and three silver pins" ~ 75,
                                 Material == "A lodestone and a pinch of dust" ~ 10,
                                 Material == "A Magnifying glass" ~ 100,
                                 Material == "A miniature platinum sword with a grip and pommel of copper and zinc, worth 250 gp" ~ 250,
                                 Material == "A small Crystal or glass cone" ~ 50,
                                 Material == "A small Crystal Sphere" ~ 100,
                                 Material == "A tiny fan and a feather of exotic origin" ~ 12,
                                 Material == "Fire and a piece of sunstone" ~ 10,
                                 Material == "Several seeds of any moonseed plant and a piece of opalescent feldspar" ~ 15,
                                 Material == "The powder of a crushed black pearl worth at least 500 gp" ~ 500,
                                TRUE~ 0
))


-The final piece of the puzzle was figuring out how I would quantify the cost of using a spell slot.

-In D&D spell slots are the expendable actions of a spellcaster in order for them to cast their spell. Since spell slots do not traditionally have a gold price to them I had to get a little creative.

-While casting spells with using spell slots is most common there are also spell scrolls available throughout the game so with this information I kept working.

-I used this site <https://blackcitadelrpg.com/spell-scroll-costs-and-rarity/> in order to make my assumptions.

-For every spell level I calculated the average gold cost and added that to my analysis.


In [ ]:
dnd_spell_value_v2<- dnd_spell_value_v1 %>%     # Putting value to spells
mutate(level_cost = case_when(Level == "0" ~  75,
                              Level == "1" ~  300,
                              Level == "2" ~  300,
                              Level == "3" ~  300,
                              Level == "4" ~  2750,
                              Level == "5" ~  2750,
                              Level == "6" ~  27500,
                              Level == "7" ~  27500,
                              Level == "8" ~  27500,
                              Level == "9" ~ 50000
))
head(dnd_spell_value_v2)

In [ ]:
cost_of_spell<- dnd_spell_value_v2 %>% 
  rowwise() %>%
  mutate(  cost_by_damage = sum(c(material_cost,level_cost/average_damage)))
head(cost_of_spell)     

In [ ]:
cost_of_spell<- cost_of_spell %>% 
  select(Name,School,"Damage type",cost_by_damage)
cost_of_spell<-unique(cost_of_spell)
min_spell_cost<-cost_of_spell %>% 
  drop_na() %>% 
  arrange(cost_by_damage)
head(min_spell_cost)

## The Wrap Up
-The conclusion is that Color Spray is the most cost efficient spell, followed closely by Fireball (which is a personal favorite).

-If you made it this far thank you for joining me on this journey!

-There are clearly several ways this could have been approached differently and I would love to hear your thoughts.


## Afterthoughts 

-If I were to approach this same objective again, it might be interesting to use less concrete assumptions, like maybe creating a range of damage rather than just the averages.

-I may return to this project in the future.
